# RAG on Azure — Day 4: Chat With Your Data + Metadata Filtering 

**Use case:** RAG over messy, mixed, real-world content (notes, meeting minutes, dated policies) where the *right* answer depends on **when**, **what category**, and **who** it came from.

- Each chunk now carries **metadata**: `category`, `author`, and `date`.
- The index defines those as **filterable / facetable** fields.
- Retrieval can apply an **OData filter** alongside the vector search — e.g. "only documents from 2024 onward", "only meeting notes" etc
- A built-in demonstration of *why this matters*: the dataset contains an **outdated 2023 remote-work policy** next to the **current 2024 one**. Without filtering, the system mixes them up. With a date filter, it answers correctly.


## 0. Install dependencies

In [14]:
%pip install -q "openai>=1.55.3" "azure-search-documents>=11.5.1" python-dotenv pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load configuration

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

explicit = Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env")
env_path = explicit if explicit.exists() else (Path(find_dotenv()) if find_dotenv() else None)

if env_path and Path(env_path).exists():
    for line in Path(env_path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
    load_dotenv(env_path, override=False)
else:
    raise FileNotFoundError("No .env file found.")

AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_SEARCH_ENDPOINT    = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY     = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME               = os.getenv("AZURE_SEARCH_INDEX_DAY4", "rag-notes-day4")

for _n, _v in [("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT),
               ("AZURE_OPENAI_API_KEY",  AZURE_OPENAI_API_KEY),
               ("AZURE_SEARCH_ENDPOINT", AZURE_SEARCH_ENDPOINT),
               ("AZURE_SEARCH_API_KEY",  AZURE_SEARCH_API_KEY)]:
    assert _v, f"Set {_n} in your .env"

print("Config OK. Using index:", INDEX_NAME)

Config OK. Using index: rag-notes-day4


## 2. Clients

In [11]:
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

aoai = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)
search_cred = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=search_cred)
print("Clients ready.")

Clients ready.


## 3. Load the messy, mixed dataset (PDFs with metadata)

Real knowledge isn't clean uniform PDFs — it's policies, meeting notes, and project updates across different sources, all with their own dates and authors. Below we load 5 PDFs from `data-day4/`. Each PDF starts with a **Document Information** block (Category, Author, Date, ...) that we parse from the extracted text so every chunk carries its metadata into the index.

Note the two remote-work policies: a **2023** version (1 day/week) and a **2024** version (3 days/week). This is the trap metadata filtering solves.

In [12]:
import re
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\data-day4")

# Parse the "Document Information" block at the top of each PDF.
def parse_metadata_block(text):
    meta = {}
    for line in text.splitlines()[:25]:               # only the top of the document
        m = re.match(r"^([A-Za-z][A-Za-z ]*?)\s*:\s*(.+)$", line.strip())
        if m:
            key = m.group(1).strip().lower()
            if key in ("category", "author", "date", "status", "project", "attendees"):
                meta[key] = m.group(2).strip()
    return meta

NOTES = []
pdfs = sorted(DATA_DIR.glob("*.pdf")) if DATA_DIR.exists() else []
print("Looking in:", DATA_DIR.resolve())

if not pdfs:
    raise FileNotFoundError(
        f"No PDFs found in {DATA_DIR.resolve()}. "
        "Place the 5 Day-4 PDFs there (or change DATA_DIR above)."
    )

for path in pdfs:
    reader = PdfReader(str(path))
    text = "\n".join((p.extract_text() or "") for p in reader.pages)
    meta = parse_metadata_block(text)

    missing = [k for k in ("category", "author", "date") if k not in meta]
    if missing:
        print(f"  WARNING {path.name}: missing metadata fields {missing}")

    NOTES.append({
        "source":   path.name,
        "category": meta.get("category", "unknown"),
        "author":   meta.get("author",   "unknown"),
        "date":     meta.get("date",     "1970-01-01"),
        "text":     text,
    })

print(f"\nLoaded {len(NOTES)} notes:")
for n in NOTES:
    print(f"  {n['source']:<40} cat={n['category']:<10} author={n['author']:<8} date={n['date']}")


Looking in: C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\data-day4

Loaded 5 notes:
  01_Remote_Work_Policy_2023.pdf           cat=policy     author=Priya    date=2023-06-01
  02_Remote_Work_Policy_2024.pdf           cat=policy     author=Priya    date=2024-02-10
  03_Q3_Planning_Meeting.pdf               cat=meeting    author=Sunil    date=2024-09-15
  04_Project_Atlas_Update.pdf              cat=project    author=Priya    date=2025-02-14
  05_Personal_Reminders.pdf                cat=personal   author=Sunil    date=2024-03-22


## 4. Chunk and carry metadata through

Each note is chunked, and every chunk inherits its note's metadata. The key idea: metadata must travel *with* the chunk all the way into the index, or you can't filter on it later.

In [13]:
def chunk_text(text, chunk_size=1000, overlap=150):
    text = text.strip()
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            window = text[start:end]
            brk = max(window.rfind("\n"), window.rfind(". "), window.rfind(" "))
            if brk > chunk_size * 0.5:
                end = start + brk + 1
        chunks.append(text[start:end].strip())
        start = max(0, end - overlap)
    return [c for c in chunks if c]

records = []
for note in NOTES:
    safe = note["source"].replace(" ", "_").replace(".", "_")
    for i, chunk in enumerate(chunk_text(note["text"])):
        records.append({
            "id": f"{safe}-{i}",
            "content": chunk,
            "source": note["source"],
            "category": note["category"],
            "author": note["author"],
            "date": note["date"] + "T00:00:00Z",   # ISO 8601 with timezone for DateTimeOffset
        })
print(f"{len(records)} chunks created. Example:")
print(records[0])

5 chunks created. Example:
{'id': '01_Remote_Work_Policy_2023_pdf-0', 'content': 'Document Information\nCategory: policy\nAuthor: Priya\nDate: 2023-06-01\nStatus: superseded\nRemote work policy (2023)\nThis policy was in effect from June 2023 through January 2024. It has since been superseded by\nthe 2024 remote work policy.\nEligibility\nRemote work was limited to 1 day per week and required director-level approval. Direct managers\ndid not have authority to approve remote arrangements on their own.\nCore hours\nEmployees working remotely were expected to be available from 9am to 4pm local time and to\nrespond to messages within 30 minutes during that window.\nEquipment\nRemote work was performed on company laptops only. Personal devices were not permitted to\naccess company systems.', 'source': '01_Remote_Work_Policy_2023.pdf', 'category': 'policy', 'author': 'Priya', 'date': '2023-06-01T00:00:00Z'}


## 5. Embed the chunks

In [14]:
def embed_texts(texts, batch_size=16):
    out = []
    for i in range(0, len(texts), batch_size):
        resp = aoai.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=texts[i:i+batch_size])
        out.extend(d.embedding for d in resp.data)
    return out

def embed(text):
    return embed_texts([text])[0]

chunk_vectors = embed_texts([r["content"] for r in records])
EMBED_DIM = len(chunk_vectors[0])
print(f"Embedded {len(chunk_vectors)} chunks. Dimension: {EMBED_DIM}")

Embedded 5 chunks. Dimension: 3072


## 6. Create the index with **filterable** metadata fields

The new part: `category`, `author`, and `date` are marked `filterable` (so we can constrain retrieval) and `facetable` (so we can count values). `date` is a `DateTimeOffset` so we can do range filters like "since 2024".

In [15]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="source",   type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="author",   type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="date",     type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index to delete (ok):", type(e).__name__)

index_client.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search))
print(f"Index '{INDEX_NAME}' created with filterable fields: category, author, date.")

Deleted existing index 'rag-notes-day4'.
Index 'rag-notes-day4' created with filterable fields: category, author, date.


## 7. Upload

In [16]:
search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_cred)

docs = [{**r, "contentVector": v} for r, v in zip(records, chunk_vectors)]
search_client.upload_documents(documents=docs)
print(f"Uploaded {len(docs)} chunks to '{INDEX_NAME}'.")

Uploaded 5 chunks to 'rag-notes-day4'.


## 8. Filtered retrieval

We add an optional OData `filter` to the vector search. A small helper builds common filters. Notice date literals are **not** quoted (they're typed as DateTimeOffset), while string values **are** quoted.

In [17]:
from azure.search.documents.models import VectorizedQuery

def build_filter(category=None, author=None, since=None, until=None):
    """since/until are ISO dates like '2024-01-01T00:00:00Z'. Returns an OData filter string or None."""
    clauses = []
    if category: clauses.append(f"category eq '{category}'")
    if author:   clauses.append(f"author eq '{author}'")
    if since:    clauses.append(f"date ge {since}")
    if until:    clauses.append(f"date lt {until}")
    return " and ".join(clauses) if clauses else None

def retrieve(query, k=5, odata_filter=None):
    vq = VectorizedQuery(vector=embed(query), k_nearest_neighbors=k, fields="contentVector")
    results = search_client.search(
        search_text=None,
        vector_queries=[vq],
        filter=odata_filter,
        select=["content", "source", "category", "author", "date"],
    )
    return [
        {"source": r["source"], "category": r["category"], "author": r["author"],
         "date": r["date"][:10], "score": r["@search.score"], "content": r["content"]}
        for r in results
    ]

## 9. The key demo: same question, different filters

Ask "how many days per week can I work remotely?" three ways. Watch the retrieved sources — without a filter, both the 2023 and 2024 policies show up (conflicting answers). Filtering to 2024-onward isolates the current policy.

In [18]:
q = "How many days per week can I work remotely?"

def show(title, hits):
    print(title)
    for h in hits:
        print(f"   [{h['score']:.3f}] {h['date']}  {h['category']:<8} {h['source']}")
    print()

show("=== No filter (mixes old + current policy) ===", retrieve(q, k=4))
show("=== Filter: only documents since 2024 ===",       retrieve(q, k=4, odata_filter=build_filter(since="2024-01-01T00:00:00Z")))
show("=== Filter: category = policy, since 2024 ===",   retrieve(q, k=4, odata_filter=build_filter(category="policy", since="2024-01-01T00:00:00Z")))

=== No filter (mixes old + current policy) ===
   [0.685] 2024-02-10  policy   02_Remote_Work_Policy_2024.pdf
   [0.675] 2023-06-01  policy   01_Remote_Work_Policy_2023.pdf
   [0.550] 2024-03-22  personal 05_Personal_Reminders.pdf
   [0.528] 2025-02-14  project  04_Project_Atlas_Update.pdf

=== Filter: only documents since 2024 ===
   [0.685] 2024-02-10  policy   02_Remote_Work_Policy_2024.pdf
   [0.550] 2024-03-22  personal 05_Personal_Reminders.pdf
   [0.528] 2025-02-14  project  04_Project_Atlas_Update.pdf
   [0.524] 2024-09-15  meeting  03_Q3_Planning_Meeting.pdf

=== Filter: category = policy, since 2024 ===
   [0.685] 2024-02-10  policy   02_Remote_Work_Policy_2024.pdf



## 10. Facets (bonus): what categories/authors exist?

Facets return value counts — useful for building filter UIs ("Policy (2), Meeting (2), Project (2)..."). Facets need a text search, so we use `search_text="*"` with `top=0` (we only want the counts, not documents).

In [19]:
facet_results = search_client.search(search_text="*", facets=["category", "author"], top=0)
facets = facet_results.get_facets()
for field, values in facets.items():
    print(f"{field}:")
    for v in values:
        print(f"   {v['value']}: {v['count']}")

author:
   Priya: 3
   Sunil: 2
category:
   policy: 2
   meeting: 1
   personal: 1
   project: 1


## 11. Answering with a filter

Generation is unchanged from Day 3 — we just thread the filter through retrieval. The same question now yields the correct, context-scoped answer.

In [20]:
def answer(question, k=5, odata_filter=None):
    hits = retrieve(question, k=k, odata_filter=odata_filter)
    if not hits:
        return "No documents match that filter."
    context = "\n\n".join(
        f"[{i+1}] (source: {h['source']}, date: {h['date']}, category: {h['category']})\n{h['content']}"
        for i, h in enumerate(hits)
    )
    system = (
        "Answer ONLY using the numbered context passages. Cite passage numbers like [1]. "
        "If the answer is not in the context, say you don't know based on the documents."
    )
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
        temperature=0,
    )
    return resp.choices[0].message.content

print("Q: How many days per week can I work remotely?\n")
print("Without filter (may be wrong / conflicted):")
print(" ", answer(q, k=4), "\n")
print("With 'since 2024' filter (current policy only):")
print(" ", answer(q, k=4, odata_filter=build_filter(since="2024-01-01T00:00:00Z")), "\n")
print("-" * 70)
print("Q: What is the status of Project Atlas? (filter: category=project)\n")
print(" ", answer("What is the status of Project Atlas?", odata_filter=build_filter(category="project")))

Q: How many days per week can I work remotely?

Without filter (may be wrong / conflicted):
  You can work remotely up to 3 days per week with manager approval, according to the current 2024 remote work policy [1]. 

With 'since 2024' filter (current policy only):
  You may work remotely up to 3 days per week with manager approval [1]. 

----------------------------------------------------------------------
Q: What is the status of Project Atlas? (filter: category=project)

  Phase 1 of Project Atlas has been completed and shipped on schedule. The customer-facing RAG assistant is now live for an internal beta cohort of 40 users [1].


## Day 4 checklist

1. Chunks carry `category`, `author`, and `date` metadata
2. Index has those as filterable / facetable fields
3. `retrieve()` accepts an OData filter and changes results accordingly
4. The "since 2024" filter isolates the current remote-work policy from the outdated one
5. Facets return value counts

### Learnings:
- **Metadata is part of relevance.** The most semantically similar chunk isn't always the right one — recency, source, and category matter.
- **Filtering vs. ranking.** A filter is a hard constraint applied alongside the soft vector ranking; combine them to scope answers precisely.
- **Real data is messy and time-sensitive.** Versioned policies are exactly where naive RAG quietly gives wrong answers.